Aim of this script: attribute an RFN line number (`code_ligne`) to each station in the
`data_chuuchuu_{data_selection}_enriched.parquet` dataset, by spatially matching each
station's coordinates against the track geometry in `geo_data/rfn_caracteristiques.gpkg`.

This is a first pass only, at the **station** level (not per train-leg / per-segment --
see the earlier discussion notebook on why that's a separate, harder problem). Three
outcomes per station:

- **no match at all** within the buffer distance -> we simply won't have infrastructure
  data for that station's rows. That's fine, left as null.
- **exactly one line matches** -> unambiguous, assign it.
- **two or more lines match** (junction stations) -> label the row `ambiguous` rather
  than guessing. Deciding between candidates is deferred to a later notebook.

In [27]:
import os
import pandas as pd
import geopandas as gpd


## Step 0 -- Load the enriched Chuuchuu dataset

In [28]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_enriched.parquet"

try:
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)
data_chuuchuu.shape


(7038209, 46)

## Step 1 -- Station coordinates & RFN line geometry

The first matching is done once per **unique station**, not once per row yet: there are ~3,800
unique stations against 7M+ stop-event rows, so we build the station-level match table
first and broadcast it back onto the full dataframe at the end.

`stations.csv`'s `db_id` column is the same identifier as `deutscheBahnStopId` in the
Chuuchuu data (confirmed earlier by spot-checking coordinates against known stations,
e.g. Paris Montparnasse).

In [29]:
unique_stations = data_chuuchuu.drop_duplicates(subset=["deutscheBahnStopId"])[
    ["deutscheBahnStopId", "stopName", "country"]
].copy()
unique_stations["db_id_str"] = unique_stations["deutscheBahnStopId"].astype(str)
print(f"{len(unique_stations)} unique stations across all countries in the dataset")


3791 unique stations across all countries in the dataset


In [30]:
data_stations = pd.read_csv("sup_data/stations.csv", sep=";", low_memory=False)
data_stations = data_stations.dropna(subset=["db_id", "latitude", "longitude"])
data_stations["db_id_str"] = data_stations["db_id"].astype("int64").astype(str)
data_stations_small = data_stations[["db_id_str", "latitude", "longitude"]].drop_duplicates(subset=["db_id_str"])

stations_with_coords = unique_stations.merge(data_stations_small, on="db_id_str", how="left")
has_coords_mask = stations_with_coords["latitude"].notna() & stations_with_coords["longitude"].notna()
print(f"{has_coords_mask.sum()} / {len(stations_with_coords)} unique stations resolved to coordinates via stations.csv")


3186 / 3791 unique stations resolved to coordinates via stations.csv


In [31]:
rfn_lines = gpd.read_file("geo_data/rfn_caracteristiques.gpkg")
rfn_lines_proj = rfn_lines.to_crs(2154)  # Lambert-93, metric CRS for a metre-based buffer
print(f"{len(rfn_lines)} track segments across {rfn_lines['code_ligne'].nunique()} lines")


1442 track segments across 639 lines


## Step 2 -- Spatial match: station -> candidate line(s)

`MATCH_BUFFER_M` is the distance (in metres) a station must fall within of a line's
track geometry to count as "on" that line. 150 m was validated earlier (checked against
50/100/300 m -- 150 m is tight enough to avoid picking up unrelated parallel lines while
still catching genuine trackside station positions).

Note this uses `sjoin` with an actual buffer polygon + `intersects`, not
`sjoin_nearest` -- `sjoin_nearest` only ever returns the single closest line, which
would silently hide every ambiguous/junction case.

In [32]:
MATCH_BUFFER_M = 150

stations_geo = stations_with_coords.loc[has_coords_mask].reset_index(drop=True)
stations_gdf = gpd.GeoDataFrame(
    stations_geo,
    geometry=gpd.points_from_xy(stations_geo["longitude"], stations_geo["latitude"]),
    crs="EPSG:4326",
).to_crs(2154)

stations_buffered = stations_gdf.copy()
stations_buffered["geometry"] = stations_buffered.geometry.buffer(MATCH_BUFFER_M)

matches = gpd.sjoin(
    stations_buffered[["db_id_str", "geometry"]],
    rfn_lines_proj[["code_ligne", "geometry"]],
    how="left",
    predicate="intersects",
)
matches = matches.drop(columns=["index_right"])
print(f"{len(matches)} (station, candidate line) pairs from {len(stations_buffered)} stations")


3494 (station, candidate line) pairs from 3186 stations


In [33]:
candidates_per_station = (
    matches.groupby("db_id_str")["code_ligne"]
    .apply(lambda s: sorted(s.dropna().unique()))
    .rename("code_ligne_candidates")
)

station_match = stations_geo[["db_id_str", "deutscheBahnStopId", "stopName", "country"]].merge(
    candidates_per_station, on="db_id_str", how="left"
)
station_match["code_ligne_candidates"] = station_match["code_ligne_candidates"].apply(
    lambda v: v if isinstance(v, list) else []
)
station_match["n_candidates"] = station_match["code_ligne_candidates"].apply(len)
station_match.head()


,db_id_str,deutscheBahnStopId,stopName,country,code_ligne_candidates,n_candidates
0,8700036,8700036,Brive-la-Gaillarde,France,[590000],1
1,8700029,8700029,Limoges Bénédictins,France,[590000],1
2,8701994,8701994,La Souterraine,France,[590000],1
3,8700010,8700010,Paris Austerlitz,France,[984000],1
4,8704839,8704839,Vierzon,France,[590000],1


## Step 3 -- Label each station: `no_coordinates` / `no_match` / `unique` / `ambiguous`

- `no_coordinates`: never resolved to a lat/lon via `stations.csv` in the first place
  (e.g. foreign stations not covered by the sup_data file, or a stale/renamed
  `deutscheBahnStopId`).
- `no_match`: coordinates exist but sit further than `MATCH_BUFFER_M` from every line
  (expected for non-French stations -- the gpkg only covers the French network -- and
  for tram/metro stops outside RFN scope).
- `unique`: exactly one candidate line -> `code_ligne` is populated.
- `ambiguous`: 2+ candidate lines -> `code_ligne` stays null, candidates kept in
  `code_ligne_candidates` for the later disambiguation step.

In [34]:
no_coords_stations = stations_with_coords.loc[~has_coords_mask, ["db_id_str", "deutscheBahnStopId", "stopName", "country"]].copy()
no_coords_stations["code_ligne_candidates"] = [[] for _ in range(len(no_coords_stations))]
no_coords_stations["n_candidates"] = 0
no_coords_stations["line_match_status"] = "no_coordinates"

def status_from_n(n):
    if n == 0:
        return "no_match"
    if n == 1:
        return "unique"
    return "ambiguous"

station_match["line_match_status"] = station_match["n_candidates"].apply(status_from_n)
station_match["code_ligne"] = station_match.apply(
    lambda r: r["code_ligne_candidates"][0] if r["line_match_status"] == "unique" else pd.NA, axis=1
)

station_line_lookup = pd.concat(
    [
        station_match[["deutscheBahnStopId", "stopName", "country", "code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"]],
        no_coords_stations.assign(code_ligne=pd.NA)[["deutscheBahnStopId", "stopName", "country", "code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"]],
    ],
    ignore_index=True,
)
assert len(station_line_lookup) == len(unique_stations)
station_line_lookup["line_match_status"].value_counts(dropna=False)


line_match_status
unique            2325
no_match           689
no_coordinates     605
ambiguous          172
Name: count, dtype: int64

### Verification -- proportions and a look at the ambiguous cases

In [35]:
counts = station_line_lookup["line_match_status"].value_counts()
pct = station_line_lookup["line_match_status"].value_counts(normalize=True) * 100
summary_table = pd.DataFrame({"n_stations": counts, "pct": pct.round(1)})
summary_table


,n_stations,pct
line_match_status,,
unique,2325,61.3
no_match,689,18.2
no_coordinates,605,16.0
ambiguous,172,4.5


In [36]:
# France-only view, since the gpkg only covers the French network -- the "no_match"
# bucket above is dominated by stations in other countries, which is expected and not
# a data quality issue
france_lookup = station_line_lookup[station_line_lookup["country"] == "France"]
print(f"France-only: {len(france_lookup)} stations")
france_lookup["line_match_status"].value_counts(normalize=True).mul(100).round(1)


France-only: 3583 stations


line_match_status
unique            64.9
no_coordinates    15.7
no_match          14.6
ambiguous          4.8
Name: proportion, dtype: float64

In [37]:
station_line_lookup[station_line_lookup["line_match_status"] == "ambiguous"].sort_values(
    "n_candidates", ascending=False
).head(20)


,deutscheBahnStopId,stopName,country,code_ligne,code_ligne_candidates,n_candidates,line_match_status
1488,8700015,Paris Saint-Lazare,France,NaN,"[334000, 334900, 340000, 973000, 975000]",5,ambiguous
55,8700023,Strasbourg,France,NaN,"[070000, 142000, 145000]",3,ambiguous
131,8700003,Les Aubrais,France,NaN,"[569000, 570000, 590000]",3,ambiguous
233,8700362,Miramas,France,NaN,"[830000, 925000, 935000]",3,ambiguous
181,8703900,Mantes-la-Jolie,France,NaN,"[334000, 340000, 366000]",3,ambiguous
849,8700469,Is-sur-Tille,France,NaN,"[838000, 843000, 849000]",3,ambiguous
1146,8701085,L'Estaque,France,NaN,"[830000, 935000, 939001]",3,ambiguous
918,8700439,Sarreguemines,France,NaN,"[159000, 161000, 163000]",3,ambiguous
612,8702030,Saint-Roch,France,NaN,"[305000, 311000, 321000]",3,ambiguous
590,8703409,Givors Canal,France,NaN,"[750000, 800000, 906000]",3,ambiguous


## Step 4 -- Broadcast the station-level match back onto every stop-event row

One row per (train, stop) in `data_chuuchuu`, so the same station's line match applies
to every train that ever stopped there.

In [38]:
rows_before = len(data_chuuchuu)

data_chuuchuu = data_chuuchuu.merge(
    station_line_lookup[["deutscheBahnStopId", "code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"]],
    on="deutscheBahnStopId",
    how="left",
)

assert len(data_chuuchuu) == rows_before, "merge changed row count -- deutscheBahnStopId isn't unique in the lookup"
data_chuuchuu["line_match_status"].value_counts(dropna=False)


line_match_status
unique            5184359
ambiguous         1013521
no_match           509134
no_coordinates     331195
Name: count, dtype: int64

In [40]:
data_chuuchuu.loc[data_chuuchuu["country"] == "France", "line_match_status"].value_counts(normalize=True).mul(100).round(1)


line_match_status
unique            79.5
ambiguous         15.5
no_coordinates     3.0
no_match           1.9
Name: proportion, dtype: float64

## Step 5 -- for trains passing through stations with several potential line matches (<i>Ambiguous</i>) : identify which lines the train took 

In [ ]:
import random

# Pick one journey (train) that stops at at least one ambiguous station, then look at
# its full itinerary -- the *other*, unambiguous stops on the same trip are usually
# enough to tell by eye which of the candidate lines the train was actually on.
ambiguous_journey_ids = data_chuuchuu.loc[
    data_chuuchuu["line_match_status"] == "ambiguous", "journey_id"
].unique()

sample_journey_id = random.choice(ambiguous_journey_ids)
sample_journey = data_chuuchuu.loc[data_chuuchuu["journey_id"] == sample_journey_id].sort_values("sort_time")

print(f"journey_id: {sample_journey_id}  ({len(sample_journey)} stops)")
sample_journey[
    [
        "stopName",
        "deutscheBahnStopId",
        "depart_terminus",
        "sort_time",
        "code_ligne",
        "code_ligne_candidates",
        "line_match_status",
    ]
]


## Exporting data

In [14]:
export_data = input("Export intermediate data to parquet? (y/n): ")

if export_data.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    # code_ligne_candidates is a list column -- parquet handles it fine via pyarrow,
    # but keep it in mind if this file is later read with an engine that doesn't
    data_chuuchuu.to_parquet(f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_lines.parquet")
